In [1]:
import os
import sys
sys.path.append(os.path.abspath('../../../'))  # Adjust path to project root
from src.app.vectorstores.search_factory import SearchServiceFactory
from src.app.config import INDEX_NAME
from src.app.vectorstores.sample_data import get_sample_documents

In [2]:
search_service = SearchServiceFactory.create_opensearch()
info = search_service.client.info()

print("Connection successful!")
print(f"Cluster name: {info['cluster_name']}")
print(f"OpenSearch version: {info['version']['number']}")
print("✓ OpenSearch Service created successfully")

Connection successful!
Cluster name: docker-cluster
OpenSearch version: 3.4.0
✓ OpenSearch Service created successfully


In [3]:
sample_docs = get_sample_documents()
print(f"Loaded {len(sample_docs)} documents")

Loaded 9 documents


In [4]:
documents_with_embeddings = []
for i, doc in enumerate(sample_docs, 1):
    print(f"  Processing {i}/{len(sample_docs)}: {doc['title'][:50]}...")
    embedding = search_service.embed(doc['content'])
    doc_with_embedding = doc.copy()
    doc_with_embedding['embedding'] = embedding
    documents_with_embeddings.append(doc_with_embedding)

  Processing 1/9: Maven UCD Build Migration Template...
  Processing 2/9: Maven UCD Publish Migration Template...
  Processing 3/9: Argo CD Build Migration Template...
  Processing 4/9: Argo CD Deployment Migration Template...
  Processing 5/9: NPM Build Migration Template...
  Processing 6/9: Docker Build and Push Migration Template...
  Processing 7/9: Gradle Build Migration Template...
  Processing 8/9: Python Build Migration Template...
  Processing 9/9: Kubernetes Deployment Migration Template...


In [5]:
# Check actual embedding dimensions
test_embedding = search_service.embed("test text")
print(f"Actual embedding dimensions: {len(test_embedding)}")
print(f"Embedding model: {search_service.embedding_model}")
print(f"The embeddings: {test_embedding}")

Actual embedding dimensions: VECTORS_DIMS
Embedding model: amazon.titan-embed-text-v1
The embeddings: [0.6484375, -0.61328125, 0.25, 0.28515625, -0.91796875, 0.1064453125, 0.341796875, -0.00152587890625, 0.6171875, -1.0078125, 0.33984375, 0.26171875, 0.80859375, 0.68359375, -0.5546875, -0.453125, -0.10595703125, -0.1806640625, -0.515625, 0.1328125, -0.62109375, 0.0260009765625, -0.28515625, 0.3671875, 0.515625, 0.5703125, 0.1962890625, -0.66015625, 0.1513671875, -0.875, 0.16796875, 0.8046875, 0.63671875, -0.734375, 0.408203125, 0.1953125, 0.5, 0.6640625, 0.39453125, 0.248046875, -0.09375, -0.71484375, 0.5234375, 0.44921875, -0.1455078125, -0.625, 0.2734375, -0.38671875, 1.25, 1.28125, -0.9921875, 1.5625, -0.06494140625, -0.1162109375, 0.203125, 1.09375, -1.0, 0.890625, 1.1171875, -0.400390625, 0.1064453125, -0.6328125, -0.0869140625, -0.45703125, 0.154296875, -0.1396484375, 0.3984375, 0.0537109375, -0.220703125, 0.625, 0.365234375, -0.022705078125, -0.123046875, 0.1064453125, -0.478515

In [6]:
if search_service.index_document(INDEX_NAME, documents_with_embeddings[0]):
    print("Single document indexed successfully")

Single document indexed successfully


In [7]:
remaining_docs = documents_with_embeddings[1:]
success = search_service.bulk_index(INDEX_NAME, remaining_docs)
print(f"✓ Bulk indexing completed: {success}")

✓ Bulk indexing completed: True


In [9]:
count = search_service.get_document_count(INDEX_NAME)
print(f"\nTotal documents in index: {count}")


Total documents in index: 18


In [ ]:
# search_service.delete_index(INDEX_NAME)